# Session 2 — Annotated Correction

## From one indicator to four — and a defensible validation report

This notebook is the post-exercise correction. It uses the same fixed scope as the student notebook and implements a broad catalogue of controls.

The controls are deliberately separated into **request**, **response**, **schema**, **scope**, **keys**, **values & definitions**, and **provenance**. They do not all have the same evidential strength.

**Session 2 boundary:** one simple response per indicator. Pagination strategies, retries, authentication, rate-limit management and completeness across multiple pages belong to Session 3.


## 0. Imports and declared contract


In [1]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 80)


In [2]:
# Fixed exercise scope: do not change it while comparing groups.
COUNTRIES = ["ESP", "POL", "MAR", "TUR"]
COUNTRY_PATH = ";".join(COUNTRIES)
START_YEAR = 2015
END_YEAR = 2024
EXPECTED_YEARS = list(range(START_YEAR, END_YEAR + 1))
EXPECTED_ROWS_PER_INDICATOR = len(COUNTRIES) * len(EXPECTED_YEARS)

BASE_URL = "https://api.worldbank.org/v2/country/{countries}/indicator/{indicator}"
PARAMS = {
    "format": "json",
    "date": f"{START_YEAR}:{END_YEAR}",
    "per_page": 100,
}

# The classroom demonstration starts with this one indicator.
DEMO_INDICATOR = "SP.POP.TOTL"

INDICATORS = {
    "SP.POP.TOTL": {
        "name": "Population, total",
        "documented_unit": "people",
    },
    "NY.GDP.PCAP.CD": {
        "name": "GDP per capita (current US$)",
        "documented_unit": "current US dollars per person",
    },
    "NY.GDP.MKTP.KD.ZG": {
        "name": "GDP growth (annual %)",
        "documented_unit": "annual percent",
    },
    "IT.NET.USER.ZS": {
        "name": "Individuals using the Internet (% of population)",
        "documented_unit": "percent of population",
    },
}

# Normal classroom mode calls the live API. For a network outage, set the
# environment variable S2_USE_LOCAL_SNAPSHOTS=1 before running the notebook.
USE_LOCAL_SNAPSHOTS = os.getenv("S2_USE_LOCAL_SNAPSHOTS", "0") == "1"
DATA_DIR = Path("Data")
OUTPUT_DIR = Path("Outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Countries:", COUNTRIES)
print("Years:", f"{START_YEAR}–{END_YEAR}")
print("Expected rows per indicator:", EXPECTED_ROWS_PER_INDICATOR)
print("Source mode:", "local snapshots" if USE_LOCAL_SNAPSHOTS else "live API")


Countries: ['ESP', 'POL', 'MAR', 'TUR']
Years: 2015–2024
Expected rows per indicator: 40
Source mode: live API


## 1. Reusable one-indicator collector and transformation


In [3]:
SNAPSHOT_FILES = {
    "SP.POP.TOTL": "world_bank_population_total_2015_2024.json",
    "NY.GDP.PCAP.CD": "world_bank_gdp_per_capita_current_usd_2015_2024.json",
    "NY.GDP.MKTP.KD.ZG": "world_bank_gdp_growth_annual_pct_2015_2024.json",
    "IT.NET.USER.ZS": "world_bank_internet_users_pct_2015_2024.json",
}


def collect_one_indicator(indicator_code: str) -> dict:
    """Collect one World Bank response using one simple GET request."""
    endpoint = BASE_URL.format(countries=COUNTRY_PATH, indicator=indicator_code)
    prepared_url = requests.Request("GET", endpoint, params=PARAMS).prepare().url
    collected_at = datetime.now(timezone.utc).replace(microsecond=0).isoformat()

    if USE_LOCAL_SNAPSHOTS:
        # Manual classroom fallback: same raw JSON shape as the live API.
        payload = json.loads((DATA_DIR / SNAPSHOT_FILES[indicator_code]).read_text(encoding="utf-8"))
        status_code = None
        called_url = prepared_url
        source_mode = "local snapshot"
    else:
        response = requests.get(endpoint, params=PARAMS, timeout=10)
        status_code = response.status_code
        called_url = response.url
        response.raise_for_status()
        payload = response.json()
        source_mode = "live API"

    return {
        "indicator_code": indicator_code,
        "endpoint": endpoint,
        "parameters": PARAMS.copy(),
        "called_url": called_url,
        "status_code": status_code,
        "source_mode": source_mode,
        "collected_at_utc": collected_at,
        "payload": payload,
    }


def observations_to_dataframe(payload: list) -> pd.DataFrame:
    """Transform payload[1] to one row per country × year × indicator."""
    observations = payload[1]
    frame = pd.json_normalize(observations).rename(
        columns={
            "indicator.id": "indicator_code",
            "indicator.value": "indicator_name",
            "countryiso3code": "country_code",
            "country.value": "country_name",
            "date": "year",
        }
    )
    columns = [
        "indicator_code", "indicator_name", "country_code", "country_name",
        "year", "value", "unit", "obs_status", "decimal",
    ]
    frame = frame[columns].copy()
    frame["year"] = pd.to_numeric(frame["year"], errors="coerce").astype("Int64")
    frame["value"] = pd.to_numeric(frame["value"], errors="coerce")
    return frame.sort_values(["indicator_code", "country_code", "year"]).reset_index(drop=True)


## 2. Phase 1 correction — generalise to four indicators

The important generalisation is small: the indicator becomes a configuration value, the same collection function is called four times, and the four tidy frames are concatenated.


In [4]:
collection_results = {}
frames = []

for indicator_code in INDICATORS:
    result = collect_one_indicator(indicator_code)
    collection_results[indicator_code] = result
    frames.append(observations_to_dataframe(result["payload"]))

all_data = pd.concat(frames, ignore_index=True)
all_data = all_data.sort_values(
    ["indicator_code", "country_code", "year"]
).reset_index(drop=True)

display(all_data.head(8))
print("Indicators collected:", all_data["indicator_code"].nunique())
print("Rows collected:", len(all_data))


,indicator_code,indicator_name,country_code,country_name,year,value,unit,obs_status,decimal
0,IT.NET.USER.ZS,Individuals using the Internet (% of population),ESP,Spain,2015,78.689632,,,0
1,IT.NET.USER.ZS,Individuals using the Internet (% of population),ESP,Spain,2016,80.561333,,,0
2,IT.NET.USER.ZS,Individuals using the Internet (% of population),ESP,Spain,2017,84.602246,,,0
3,IT.NET.USER.ZS,Individuals using the Internet (% of population),ESP,Spain,2018,86.107236,,,0
4,IT.NET.USER.ZS,Individuals using the Internet (% of population),ESP,Spain,2019,90.718665,,,0
5,IT.NET.USER.ZS,Individuals using the Internet (% of population),ESP,Spain,2020,93.205649,,,0
6,IT.NET.USER.ZS,Individuals using the Internet (% of population),ESP,Spain,2021,93.897522,,,0
7,IT.NET.USER.ZS,Individuals using the Internet (% of population),ESP,Spain,2022,94.485543,,,0


Indicators collected: 4
Rows collected: 160


In [5]:
# Keep the raw responses and the request evidence before further transformation.
provenance_records = []

for indicator_code, result in collection_results.items():
    safe_code = indicator_code.replace(".", "_")
    raw_path = OUTPUT_DIR / f"raw_{safe_code}_2015_2024.json"
    provenance_path = OUTPUT_DIR / f"provenance_{safe_code}_2015_2024.json"

    raw_path.write_text(
        json.dumps(result["payload"], ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    provenance = {
        "source": "World Bank Indicators API V2",
        "indicator_code": indicator_code,
        "endpoint": result["endpoint"],
        "parameters": result["parameters"],
        "called_url": result["called_url"],
        "status_code": result["status_code"],
        "source_mode": result["source_mode"],
        "collected_at_utc": result["collected_at_utc"],
        "countries": COUNTRIES,
        "years": [START_YEAR, END_YEAR],
        "raw_file": str(raw_path),
    }
    provenance_path.write_text(
        json.dumps(provenance, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    provenance_records.append({**provenance, "provenance_file": str(provenance_path)})

provenance_df = pd.DataFrame(provenance_records)
display(provenance_df[["indicator_code", "source_mode", "status_code", "raw_file"]])


,indicator_code,source_mode,status_code,raw_file
0,SP.POP.TOTL,live API,200,Outputs/raw_SP_POP_TOTL_2015_2024.json
1,NY.GDP.PCAP.CD,live API,200,Outputs/raw_NY_GDP_PCAP_CD_2015_2024.json
2,NY.GDP.MKTP.KD.ZG,live API,200,Outputs/raw_NY_GDP_MKTP_KD_ZG_2015_2024.json
3,IT.NET.USER.ZS,live API,200,Outputs/raw_IT_NET_USER_ZS_2015_2024.json


## 3. Control framework

Each control records an observed result, an expectation, a verdict and an evidence strength.

- **Strong:** directly compares the declared contract with exact returned keys or definitions.
- **Medium:** checks structure, counts, types or internal consistency.
- **Weak:** useful as a warning signal, but cannot prove correctness alone.


In [6]:
control_results = []


def record_control(category, control, passed, observed, expected, strength, rationale, warning=False):
    """Append one readable control result."""
    if warning:
        verdict = "WARNING" if not passed else "PASS"
    else:
        verdict = "PASS" if passed else "FAIL"
    control_results.append(
        {
            "category": category,
            "control": control,
            "verdict": verdict,
            "strength": strength,
            "observed": str(observed),
            "expected": str(expected),
            "rationale": rationale,
        }
    )


def record_review(category, control, observed, expected, strength, rationale):
    """Append a control that requires human review rather than a binary assertion."""
    control_results.append(
        {
            "category": category,
            "control": control,
            "verdict": "REVIEW",
            "strength": strength,
            "observed": str(observed),
            "expected": str(expected),
            "rationale": rationale,
        }
    )


### A. Request and source controls


In [7]:
requested_codes = set(INDICATORS)
collected_codes = set(collection_results)

record_control(
    "Request & source", "Four indicator requests were executed",
    len(collection_results) == 4, len(collection_results), 4, "medium",
    "Confirms that the loop ran four times; does not validate the content.",
)
record_control(
    "Request & source", "Requested indicator codes match the declared contract",
    collected_codes == requested_codes, sorted(collected_codes), sorted(requested_codes), "strong",
    "An exact set comparison detects an omitted or substituted indicator.",
)
record_control(
    "Request & source", "Every endpoint contains its requested indicator code",
    all(code in result["endpoint"] for code, result in collection_results.items()),
    {code: result["endpoint"] for code, result in collection_results.items()},
    "indicator code embedded in each endpoint", "strong",
    "Links each response to the intended resource path.",
)
record_control(
    "Request & source", "Every endpoint contains the four explicit country codes",
    all(COUNTRY_PATH in result["endpoint"] for result in collection_results.values()),
    COUNTRY_PATH, COUNTRY_PATH, "strong",
    "Avoids silently including World Bank aggregates or unintended countries.",
)
record_control(
    "Request & source", "Output format is JSON",
    all(result["parameters"].get("format") == "json" for result in collection_results.values()),
    {code: result["parameters"].get("format") for code, result in collection_results.items()},
    "json", "medium", "Confirms the representation requested from the API.",
)
record_control(
    "Request & source", "Requested period is exactly 2015:2024",
    all(result["parameters"].get("date") == "2015:2024" for result in collection_results.values()),
    {code: result["parameters"].get("date") for code, result in collection_results.items()},
    "2015:2024", "strong", "Validates the requested time scope before inspecting returned rows.",
)
record_control(
    "Request & source", "per_page can contain the 40 expected observations",
    all(int(result["parameters"]["per_page"]) >= EXPECTED_ROWS_PER_INDICATOR for result in collection_results.values()),
    PARAMS["per_page"], f">= {EXPECTED_ROWS_PER_INDICATOR}", "medium",
    "Sufficient for this contained response only; it is not a general pagination solution.",
)
record_control(
    "Request & source", "Collection timestamp is recorded for every result",
    all(bool(result["collected_at_utc"]) for result in collection_results.values()),
    [result["collected_at_utc"] for result in collection_results.values()],
    "one non-empty UTC timestamp per indicator", "medium", "Supports reproducibility and revision tracking.",
)
record_control(
    "Request & source", "Source mode is explicit for every result",
    all(result["source_mode"] in {"live API", "local snapshot"} for result in collection_results.values()),
    sorted({result["source_mode"] for result in collection_results.values()}),
    "live API or local snapshot", "medium", "Prevents a fallback response from being mistaken for a live call.",
)

if USE_LOCAL_SNAPSHOTS:
    record_review(
        "Request & source", "HTTP status check",
        "not applicable: local snapshots used", "200 for every live request", "medium",
        "A saved response has no live HTTP status; the source mode must remain visible.",
    )
else:
    record_control(
        "Request & source", "Every live HTTP status is 200",
        all(result["status_code"] == 200 for result in collection_results.values()),
        {code: result["status_code"] for code, result in collection_results.items()},
        "200 for every request", "medium", "A 200 permits inspection; it does not validate the data content.",
    )


### B. Response-structure and metadata controls


In [8]:
for indicator_code, result in collection_results.items():
    payload = result["payload"]
    metadata, observations = payload

    record_control(
        "Response", f"{indicator_code}: top level is a two-part list",
        isinstance(payload, list) and len(payload) == 2,
        f"{type(payload).__name__}, length {len(payload)}", "list, length 2", "strong",
        "Confirms the World Bank [metadata, observations] contract.",
    )
    record_control(
        "Response", f"{indicator_code}: metadata and observations have expected types",
        isinstance(metadata, dict) and isinstance(observations, list),
        f"{type(metadata).__name__} + {type(observations).__name__}", "dict + list", "medium",
        "Prevents flattening the wrong part of the response.",
    )
    record_control(
        "Response", f"{indicator_code}: metadata total equals raw observation count",
        int(metadata["total"]) == len(observations),
        f"total={metadata['total']}; raw={len(observations)}", "equal", "medium",
        "Internal response consistency check.",
    )
    record_control(
        "Response", f"{indicator_code}: metadata reports one page for this exercise",
        int(metadata["page"]) == 1 and int(metadata["pages"]) == 1,
        f"page={metadata['page']}; pages={metadata['pages']}", "page=1; pages=1", "medium",
        "Valid only for this small contained request; multi-page collection belongs to Session 3.",
    )
    record_control(
        "Response", f"{indicator_code}: metadata total is 40",
        int(metadata["total"]) == EXPECTED_ROWS_PER_INDICATOR,
        metadata["total"], EXPECTED_ROWS_PER_INDICATOR, "medium",
        "Useful count evidence, but exact keys are stronger.",
    )
    record_control(
        "Response", f"{indicator_code}: per_page is at least total",
        int(metadata["per_page"]) >= int(metadata["total"]),
        f"per_page={metadata['per_page']}; total={metadata['total']}", "per_page >= total", "medium",
        "Confirms that this response can fit on one page.",
    )

raw_required_paths = ["indicator", "country", "countryiso3code", "date", "value", "unit", "obs_status", "decimal"]
missing_raw_fields = {}
for indicator_code, result in collection_results.items():
    observations = result["payload"][1]
    missing_raw_fields[indicator_code] = sorted(
        field for field in raw_required_paths
        if any(field not in row for row in observations)
    )

record_control(
    "Response", "Required raw observation fields are present",
    all(not fields for fields in missing_raw_fields.values()),
    missing_raw_fields, "no missing raw fields", "strong",
    "Schema validation detects structural changes before transformation.",
)


### C. DataFrame schema and declared grain


In [9]:
expected_columns = {
    "indicator_code", "indicator_name", "country_code", "country_name",
    "year", "value", "unit", "obs_status", "decimal",
}
observed_columns = set(all_data.columns)

record_control(
    "Schema & grain", "Flattened columns match the declared schema",
    observed_columns == expected_columns, sorted(observed_columns), sorted(expected_columns), "strong",
    "Detects missing, renamed or unintended analytical fields.",
)
record_control(
    "Schema & grain", "Year is stored as an integer-like field",
    pd.api.types.is_integer_dtype(all_data["year"].dtype), all_data["year"].dtype, "integer dtype", "medium",
    "Prevents lexicographic year logic and makes the grain explicit.",
)
record_control(
    "Schema & grain", "Value is numeric",
    pd.api.types.is_numeric_dtype(all_data["value"].dtype), all_data["value"].dtype, "numeric dtype", "medium",
    "Required before arithmetic, comparisons and range checks.",
)

key_fields = ["indicator_code", "country_code", "year"]
missing_key_cells = int(all_data[key_fields].isna().sum().sum())
record_control(
    "Schema & grain", "No key field is missing",
    missing_key_cells == 0, missing_key_cells, 0, "strong",
    "A row without its indicator, country or year cannot be matched to the declared grain.",
)


### D. Exact scope controls


In [10]:
observed_indicators = set(all_data["indicator_code"].dropna())
observed_countries = set(all_data["country_code"].dropna())
observed_years = set(all_data["year"].dropna().astype(int))
expected_years = set(EXPECTED_YEARS)

record_control(
    "Scope", "Returned indicators exactly match the four requested indicators",
    observed_indicators == requested_codes, sorted(observed_indicators), sorted(requested_codes), "strong",
    "Exact set equality catches missing and unexpected indicators.",
)
record_control(
    "Scope", "Returned countries exactly match the four requested countries",
    observed_countries == set(COUNTRIES), sorted(observed_countries), sorted(COUNTRIES), "strong",
    "Exact set equality catches a missing country or an unwanted aggregate.",
)
record_control(
    "Scope", "Returned years exactly match 2015–2024",
    observed_years == expected_years, sorted(observed_years), EXPECTED_YEARS, "strong",
    "Exact set equality is stronger than checking only minimum and maximum year.",
)
record_control(
    "Scope", "Combined row count is 160",
    len(all_data) == 160, len(all_data), 160, "medium",
    "A useful reconciliation, but the right count can still contain wrong keys.",
)

rows_per_indicator = all_data.groupby("indicator_code").size().to_dict()
record_control(
    "Scope", "Each indicator has 40 rows",
    all(count == EXPECTED_ROWS_PER_INDICATOR for count in rows_per_indicator.values())
    and set(rows_per_indicator) == requested_codes,
    rows_per_indicator, f"40 for each of {sorted(requested_codes)}", "medium",
    "Localises an incomplete indicator response.",
)

rows_per_series = all_data.groupby(["indicator_code", "country_code"]).size()
record_control(
    "Scope", "Every country–indicator series has ten rows",
    rows_per_series.eq(10).all() and len(rows_per_series) == 16,
    rows_per_series.value_counts().sort_index().to_dict(), "16 series × 10 rows", "medium",
    "Detects uneven series lengths, but exact year keys remain stronger.",
)


### E. Exact key controls


In [11]:
expected_keys = {
    (indicator, country, year)
    for indicator in INDICATORS
    for country in COUNTRIES
    for year in EXPECTED_YEARS
}
observed_keys = set(
    all_data[["indicator_code", "country_code", "year"]]
    .dropna()
    .assign(year=lambda table: table["year"].astype(int))
    .itertuples(index=False, name=None)
)
missing_keys = expected_keys - observed_keys
unexpected_keys = observed_keys - expected_keys
duplicate_keys = int(all_data.duplicated(["indicator_code", "country_code", "year"]).sum())

record_control(
    "Exact keys", "No expected indicator–country–year key is missing",
    len(missing_keys) == 0, len(missing_keys), 0, "strong",
    "Directly compares the returned analytical grain with the full declared contract.",
)
record_control(
    "Exact keys", "No unexpected indicator–country–year key is present",
    len(unexpected_keys) == 0, len(unexpected_keys), 0, "strong",
    "Detects extra countries, years or indicators even when the row count looks correct.",
)
record_control(
    "Exact keys", "No indicator–country–year key is duplicated",
    duplicate_keys == 0, duplicate_keys, 0, "strong",
    "Prevents double counting and many-to-many joins at the declared grain.",
)
record_control(
    "Exact keys", "Observed unique-key count equals expected-key count",
    len(observed_keys) == len(expected_keys), len(observed_keys), len(expected_keys), "strong",
    "Reconciles exact coverage after missing, unexpected and duplicate checks.",
)


### F. Values, labels and units


In [12]:
missing_values = int(all_data["value"].isna().sum())
finite_values = int(np.isfinite(all_data["value"].dropna()).sum())

record_control(
    "Values & definitions", "No observation value is missing",
    missing_values == 0, missing_values, 0, "medium",
    "Missingness changes what comparisons can be made; zero missing does not prove correctness.",
)
record_control(
    "Values & definitions", "All non-missing values are finite",
    finite_values == all_data["value"].notna().sum(), finite_values,
    int(all_data["value"].notna().sum()), "medium", "Rejects infinite values that would distort summaries.",
)

returned_names = (
    all_data[["indicator_code", "indicator_name"]]
    .drop_duplicates()
    .set_index("indicator_code")["indicator_name"]
    .to_dict()
)
expected_names = {code: details["name"] for code, details in INDICATORS.items()}
record_control(
    "Values & definitions", "Returned indicator names match the documented definitions",
    returned_names == expected_names, returned_names, expected_names, "strong",
    "The code and label together reduce the risk of interpreting the wrong construct.",
)

code_name_counts = all_data.groupby("indicator_code")["indicator_name"].nunique()
record_control(
    "Values & definitions", "Each indicator code maps to exactly one name",
    code_name_counts.eq(1).all(), code_name_counts.to_dict(), "one name per code", "medium",
    "Detects inconsistent labels within the combined table.",
)

country_name_counts = all_data.groupby("country_code")["country_name"].nunique()
record_control(
    "Values & definitions", "Each country code maps to exactly one country name",
    country_name_counts.eq(1).all(), country_name_counts.to_dict(), "one name per code", "medium",
    "Detects inconsistent geography labels.",
)

documented_units = {code: details["documented_unit"] for code, details in INDICATORS.items()}
record_control(
    "Values & definitions", "A documented unit is declared for every indicator",
    set(documented_units) == requested_codes and all(documented_units.values()),
    documented_units, "one documented unit per indicator", "strong",
    "The generic API unit field may be empty, so the indicator definition is the authoritative contract.",
)

api_unit_values = sorted(all_data["unit"].fillna("").unique())
record_review(
    "Values & definitions", "Generic API unit field inspected",
    api_unit_values, "interpret with the documented indicator unit", "weak",
    "An empty generic unit field is not evidence that the indicator is unitless.",
)

range_checks = {
    "SP.POP.TOTL": (all_data.loc[all_data["indicator_code"] == "SP.POP.TOTL", "value"] > 0).all(),
    "NY.GDP.PCAP.CD": (all_data.loc[all_data["indicator_code"] == "NY.GDP.PCAP.CD", "value"] > 0).all(),
    "IT.NET.USER.ZS": all_data.loc[all_data["indicator_code"] == "IT.NET.USER.ZS", "value"].between(0, 100).all(),
    "NY.GDP.MKTP.KD.ZG": all_data.loc[all_data["indicator_code"] == "NY.GDP.MKTP.KD.ZG", "value"].between(-50, 50).all(),
}
record_control(
    "Values & definitions", "Indicator-specific plausibility screens pass",
    all(range_checks.values()), range_checks,
    "population > 0; GDP per capita > 0; Internet 0–100; GDP growth −50 to 50", "weak",
    "Range screens can flag impossible values but cannot prove that a plausible value is correct.",
    warning=True,
)

latest_counts = all_data.loc[all_data["year"] == END_YEAR].groupby("indicator_code")["country_code"].nunique()
record_control(
    "Values & definitions", "The latest requested year exists for every country and indicator",
    latest_counts.eq(len(COUNTRIES)).all() and len(latest_counts) == len(INDICATORS),
    latest_counts.to_dict(), "4 countries for each indicator in 2024", "medium",
    "Prevents a latest-year comparison built from uneven coverage.",
)


### G. Reconciliation, provenance and review evidence


In [13]:
metadata_total_sum = sum(int(result["payload"][0]["total"]) for result in collection_results.values())
record_control(
    "Reconciliation & provenance", "Sum of metadata totals equals combined DataFrame rows",
    metadata_total_sum == len(all_data), metadata_total_sum, len(all_data), "medium",
    "Reconciles raw response metadata with the transformed result.",
)

raw_file_exists = provenance_df["raw_file"].map(lambda path: Path(path).exists())
provenance_file_exists = provenance_df["provenance_file"].map(lambda path: Path(path).exists())
record_control(
    "Reconciliation & provenance", "One raw JSON file was preserved per indicator",
    raw_file_exists.all() and len(raw_file_exists) == 4,
    int(raw_file_exists.sum()), 4, "strong", "Preserves evidence before transformation.",
)
record_control(
    "Reconciliation & provenance", "One provenance file was preserved per indicator",
    provenance_file_exists.all() and len(provenance_file_exists) == 4,
    int(provenance_file_exists.sum()), 4, "strong", "Links each dataset to URL, parameters, time and source mode.",
)

fingerprints = (
    all_data.groupby("indicator_code")["value"]
    .agg(rows="size", non_missing="count", minimum="min", maximum="max", mean="mean", total="sum")
    .round(4)
)
record_review(
    "Reconciliation & provenance", "Per-indicator value fingerprints reviewed",
    fingerprints.to_dict(orient="index"), "stable enough to explain; investigate unexpected changes", "weak",
    "Fingerprints help compare runs but can change after legitimate World Bank revisions.",
)

latest_values = (
    all_data.loc[all_data["year"] == END_YEAR,
                 ["indicator_code", "country_code", "year", "value"]]
    .sort_values(["indicator_code", "country_code"])
)
record_review(
    "Reconciliation & provenance", "Latest-year values inspected by indicator and country",
    f"{len(latest_values)} rows displayed below", "16 values with country, year and unit understood", "weak",
    "Human review can reveal surprising values, but visual plausibility is not proof.",
)

display(fingerprints)
display(latest_values)


,rows,non_missing,minimum,maximum,mean,total
indicator_code,,,,,,
IT.NET.USER.ZS,40,40,5.374500e+01,9.575750e+01,8.018000e+01,3.207201e+03
NY.GDP.MKTP.KD.ZG,40,40,-1.094010e+01,1.181110e+01,3.359900e+00,1.343965e+02
NY.GDP.PCAP.CD,40,40,3.140856e+03,3.532677e+04,1.540158e+04,6.160631e+05
SP.POP.TOTL,40,40,3.460759e+07,8.551866e+07,5.091305e+07,2.036522e+09


,indicator_code,country_code,year,value
9,IT.NET.USER.ZS,ESP,2024,9.575745e+01
19,IT.NET.USER.ZS,MAR,2024,9.120000e+01
29,IT.NET.USER.ZS,POL,2024,8.858496e+01
39,IT.NET.USER.ZS,TUR,2024,8.730788e+01
49,NY.GDP.MKTP.KD.ZG,ESP,2024,3.455254e+00
59,NY.GDP.MKTP.KD.ZG,MAR,2024,3.793365e+00
69,NY.GDP.MKTP.KD.ZG,POL,2024,3.028416e+00
79,NY.GDP.MKTP.KD.ZG,TUR,2024,3.327623e+00
89,NY.GDP.PCAP.CD,ESP,2024,3.532677e+04
99,NY.GDP.PCAP.CD,MAR,2024,4.153194e+03


## 4. Consolidated control report and verdict


In [14]:
control_report = pd.DataFrame(control_results)
category_order = [
    "Request & source", "Response", "Schema & grain", "Scope", "Exact keys",
    "Values & definitions", "Reconciliation & provenance",
]
control_report["category"] = pd.Categorical(
    control_report["category"], categories=category_order, ordered=True
)
control_report = control_report.sort_values(["category", "strength", "control"]).reset_index(drop=True)

display(control_report[["category", "control", "verdict", "strength", "observed", "expected"]])
display(pd.crosstab(control_report["category"], control_report["verdict"], margins=True))

failed_controls = control_report.loc[control_report["verdict"] == "FAIL"]
certification = "CERTIFIED" if failed_controls.empty else "NOT YET CERTIFIED"

print("Distinct controls:", len(control_report))
print("Failed controls:", len(failed_controls))
print("Verdict for the declared 4 × 4 × 10 scope:", certification)

assert failed_controls.empty, "At least one mandatory validation control failed."


,category,control,verdict,strength,observed,expected
0,Request & source,Collection timestamp is recorded for every result,PASS,medium,"['2026-09-17T08:50:12+00:00', '2026-09-17T08:50:12+00:00', '2026-09-17T08:50...",one non-empty UTC timestamp per indicator
1,Request & source,Every live HTTP status is 200,PASS,medium,"{'SP.POP.TOTL': 200, 'NY.GDP.PCAP.CD': 200, 'NY.GDP.MKTP.KD.ZG': 200, 'IT.NE...",200 for every request
2,Request & source,Four indicator requests were executed,PASS,medium,4,4
3,Request & source,Output format is JSON,PASS,medium,"{'SP.POP.TOTL': 'json', 'NY.GDP.PCAP.CD': 'json', 'NY.GDP.MKTP.KD.ZG': 'json...",json
4,Request & source,Source mode is explicit for every result,PASS,medium,['live API'],live API or local snapshot
...,...,...,...,...,...,...
58,Reconciliation & provenance,Sum of metadata totals equals combined DataFrame rows,PASS,medium,160,160
59,Reconciliation & provenance,One provenance file was preserved per indicator,PASS,strong,4,4
60,Reconciliation & provenance,One raw JSON file was preserved per indicator,PASS,strong,4,4
61,Reconciliation & provenance,Latest-year values inspected by indicator and country,REVIEW,weak,16 rows displayed below,"16 values with country, year and unit understood"


verdict,PASS,REVIEW,All
category,,,
Request & source,10,0,10
Response,25,0,25
Schema & grain,4,0,4
Scope,6,0,6
Exact keys,4,0,4
Values & definitions,8,1,9
Reconciliation & provenance,3,2,5
All,60,3,63


Distinct controls: 63
Failed controls: 0
Verdict for the declared 4 × 4 × 10 scope: CERTIFIED


## 5. Which evidence is strongest?

The strongest controls are not necessarily the longest or most technical. For this exercise, the most discriminating evidence is:

1. **exact returned indicator set** versus the four requested codes;
2. **exact returned country set** versus `ESP`, `POL`, `MAR`, `TUR`;
3. **exact returned year set** versus 2015–2024;
4. **exact 160-key comparison** at `indicator_code × country_code × year`, including missing, unexpected and duplicate keys;
5. **indicator code + official label + documented unit**;
6. **preserved raw JSON and provenance**.

Row counts, summary statistics and plausible ranges remain useful, but they are supporting evidence. A wrong dataset can have 160 rows and plausible values.


## 6. Model group presentation

- **Modification:** the indicator became a configuration dictionary; one loop collected and concatenated four responses.
- **Result:** 4 indicators × 4 countries × 10 years = **160 expected keys**.
- **Strongest proof:** the observed set of `indicator × country × year` keys exactly equals the expected set; no key is missing, unexpected or duplicated.
- **Definitions:** all four returned codes and names match the declared indicators; units are taken from indicator documentation, not inferred from the empty generic `unit` field.
- **Weak evidence:** row counts and plausible ranges can identify problems but cannot prove correctness alone.
- **Verdict:** **CERTIFIED for the declared Session 2 scope**, provided all mandatory controls pass. This is not yet a robust multi-page production collector.


## 7. Transition to Session 3

This correction validates four small, contained responses. It does not solve:

- multi-page collection;
- authentication and secret management;
- rate limits;
- retries and backoff;
- caching;
- proof of completeness across pages.

**Next session: From an API Call to a Reliable Collection.**
